# Outline Artifact Viewer (By Run Timestamp)

This notebook prints outline artifacts for a specific run timestamp, grouped by attempts, expansions, revisions, preliminary plan drafts/refinements, and the plan. Each section includes the exact system prompt and the user prompt template used by the LLM for that step, with placeholders where run-specific data is injected.

In [3]:
from pathlib import Path
import json
import re

ARTIFACTS_DIR = Path("artifacts")

# Optional: set the run timestamp you want to inspect (from the filename prefix).
# Example: "2026_02_25_02_07_05"
RUN_TIMESTAMP = None

# Optional: set RUN_ID if multiple runs share the same timestamp.
RUN_ID = None  # e.g., 6

# If RUN_TIMESTAMP is None, pick the most recent outline artifact automatically.
AUTO_PICK_LATEST = True


def _run_prefix_from_path(path: Path) -> str | None:
    match = re.match(
        r"^(\d{4}_\d{2}_\d{2}_\d{2}_\d{2}_\d{2})_outline_run_(\d+)_outline_",
        path.name,
    )
    if not match:
        return None
    return f"{match.group(1)}_outline_run_{match.group(2)}"


def _latest_outline_artifact() -> Path | None:
    patterns = [
        "*_outline_run_*_outline_revision*.json",
        "*_outline_run_*_outline_expand*.json",
        "*_outline_run_*_outline_attempt*.json",
    ]
    candidates = []
    for pattern in patterns:
        candidates.extend(ARTIFACTS_DIR.glob(pattern))
    if not candidates:
        return None
    return max(candidates, key=lambda p: p.stat().st_mtime)


def _pick_run_prefix():
    if RUN_ID is not None and RUN_TIMESTAMP:
        return f"{RUN_TIMESTAMP}_outline_run_{RUN_ID}"

    if not RUN_TIMESTAMP and AUTO_PICK_LATEST:
        latest = _latest_outline_artifact()
        if latest:
            return _run_prefix_from_path(latest)

    if not RUN_TIMESTAMP:
        return None

    candidates = list(ARTIFACTS_DIR.glob(f"{RUN_TIMESTAMP}_outline_run_*_outline_*.json"))
    prefixes = sorted({_run_prefix_from_path(p) for p in candidates} - {None})
    if len(prefixes) == 1:
        return prefixes[0]
    if len(prefixes) > 1:
        latest = max(candidates, key=lambda p: p.stat().st_mtime)
        return _run_prefix_from_path(latest)
    return None


def _print_outline_json(path: Path | None, label: str):
    print(f"=== {label} ===")
    if not path:
        print("(not found)")
        return None
    print(f"File: {path}")
    data = json.loads(path.read_text(encoding="utf-8"))
    outline = data.get("outline")
    if isinstance(outline, list):
        for idx, item in enumerate(outline, 1):
            print(f"{idx}. {item}")
    else:
        print(json.dumps(data, indent=2, ensure_ascii=False))
    return outline if isinstance(outline, list) else None


def _print_outline_group(paths, label):
    print(f"=== {label} ===")
    if not paths:
        print("(not found)")
        return
    for path in paths:
        _print_outline_json(path, path.name)


def _print_markdown(path: Path | None, label: str):
    print(f"=== {label} ===")
    if not path:
        print("(not found)")
        return
    print(f"File: {path}")
    print(path.read_text(encoding="utf-8"))


RUN_PREFIX = _pick_run_prefix()
print(f"RUN_PREFIX: {RUN_PREFIX}")



def _plan_run_prefix_from_path(path: Path) -> str | None:
    match = re.match(
        r"^(\d{4}_\d{2}_\d{2}_\d{2}_\d{2}_\d{2})_plan_run_(\d+)_plan_",
        path.name,
    )
    if not match:
        return None
    return f"{match.group(1)}_plan_run_{match.group(2)}"


def _latest_plan_artifact() -> Path | None:
    patterns = [
        "*_plan_run_*_plan_draft_round*.json",
        "*_plan_run_*_plan_refine_round*.json",
        "*_plan_draft_round*.json",
        "*_plan_refine_round*.json",
    ]
    candidates = []
    for pattern in patterns:
        candidates.extend(ARTIFACTS_DIR.glob(pattern))
    if not candidates:
        return None
    return max(candidates, key=lambda p: p.stat().st_mtime)


def _pick_plan_prefix(outline_prefix: str | None = None):
    if RUN_ID is not None and RUN_TIMESTAMP:
        return f"{RUN_TIMESTAMP}_plan_run_{RUN_ID}"

    if outline_prefix:
        match = re.match(r"^(\d{4}_\d{2}_\d{2}_\d{2}_\d{2}_\d{2})_outline_run_(\d+)", outline_prefix)
        if match:
            return f"{match.group(1)}_plan_run_{match.group(2)}"

    if not RUN_TIMESTAMP and AUTO_PICK_LATEST:
        latest = _latest_plan_artifact()
        if latest:
            return _plan_run_prefix_from_path(latest)

    if not RUN_TIMESTAMP:
        return None

    candidates = list(ARTIFACTS_DIR.glob(f"{RUN_TIMESTAMP}_plan_run_*_plan_*.json"))
    prefixes = sorted({_plan_run_prefix_from_path(p) for p in candidates} - {None})
    if len(prefixes) == 1:
        return prefixes[0]
    if len(prefixes) > 1:
        latest = max(candidates, key=lambda p: p.stat().st_mtime)
        return _plan_run_prefix_from_path(latest)
    return None


def _print_plan_json(path: Path | None, label: str):
    print(f"=== {label} ===")
    if not path:
        print("(not found)")
        return None
    print(f"File: {path}")
    data = json.loads(path.read_text(encoding="utf-8"))
    print(json.dumps(data, indent=2, ensure_ascii=False))
    return data


def _print_plan_group(paths, label):
    print(f"=== {label} ===")
    if not paths:
        print("(not found)")
        return
    for path in paths:
        _print_plan_json(path, path.name)


RUN_PREFIX: 2026_02_25_14_11_09_outline_run_16


## Outline Attempts

Input is the topic, methods, selected documents, and KG fact cards assembled during the run. The outliner triggers attempts when it first tries to generate a compliant outline; each attempt is a fresh LLM call saved immediately after parsing. Attempts occur before any length-expansion or revision logic is applied.

System prompt used by the LLM:
```text
You are a research planning agent. Produce a research plan (not a report outline). All natural-language content MUST be in the requested output language.
Return ONE valid JSON object only. No markdown, no commentary, no extra text.

Requirements:
- Must include key: outline
- outline MUST be a non-empty array (8-12 major steps).
- Each major step must include 3-5 substeps.
- Each substep should be 2-3 sentences.
- Optional sub-substeps are allowed when helpful.
- Use imperative phrasing (e.g., "Search...", "Compare...", "Investigate...").
- Do not write report section headings.
- Total outline length MUST be 1000-2000 words across titles and substeps in the output language.
- Output must be strict JSON (double quotes, no trailing commas)
- Treat methods as analysis approaches to apply to the topic, not as the topic itself.
- If methods are provided, the research plan MUST explicitly structure steps around those methods.
- All natural-language content MUST be in the requested output language.
- Keep JSON keys in English.
- Keep paper titles, dataset names, benchmarks, model names, APIs, and acronyms in English.
- Use ASCII punctuation for JSON (":" and ","), not full-width punctuation.
- When evidence includes confidence or recency, prefer higher-confidence and more recent evidence.
All natural-language content MUST be in the requested output language.

Example (Chinese):
{"outline":[{"title":"查找官方技术报告与发布博客以提取架构与训练细节。","substeps":["列出要检索的官方来源与发布页面。","提取训练规模、数据来源与能力摘要。",{"text":"捕捉部署背景与版本历史。","substeps":["记录发布节奏与更新日志。","整理公开的限制与注意事项。"]}]},{"title":"比较不同模型的上下文长度与长上下文准确性。","substeps":["收集厂商声明与独立基准。","分析长上下文召回与检索性能。",{"text":"识别失效案例。","substeps":["总结常见错误模式。","记录实践中的缓解策略。"]}]}]}

Example (English):
{"outline":[{"title":"Search for official technical reports and release blogs to extract architecture and training details.","substeps":["List official sources and release pages to target.","Extract training scale, data sources, and capability summaries.",{"text":"Capture deployment context and version history.","substeps":["Note release cadence and changelogs.","Record published caveats or limitations."]}]},{"title":"Compare context window sizes and long-context accuracy across models.","substeps":["Collect vendor claims and independent benchmarks.","Analyze long-context recall and retrieval performance.",{"text":"Identify failure cases.","substeps":["Summarize common error patterns.","Note mitigation strategies reported by practitioners."]}]}]}
```

User prompt template sent to the LLM:
```json
{
  "topic": "<topic>",
  "interests": ["<interest>", "..."],
  "methods": ["<method>", "..."],
  "documents": [
    {"title": "<title>", "source_type": "<type>", "source": "<source>", "doc_id": "<id>"}
  ],
  "keywords": ["<keyword>", "..."],
  "output_language": "<language>",
  "instructions": {
    "output_json_schema": {"outline": ["string"]},
    "language_policy": [
      "All natural-language content MUST be in <language>.",
      "Keep paper titles, dataset names, benchmarks, model names, APIs, and acronyms in English.",
      "All natural-language content MUST be in <language>."
    ],
    "outline_expectations": [
      "Include major sections and subtopics to expand later.",
      "Provide 8-12 major steps.",
      "Each major step must include 3-5 substeps.",
      "Each substep should be 2-3 sentences.",
      "List source-backed bullets when available.",
      "Emphasize gaps and future research directions.",
      "Length must be 1000-2000 words in the output language."
    ],
    "method_guidance": [
      "Use methods as analytical lenses applied to the topic and evidence.",
      "Do not treat methods as the topic itself.",
      "If methods are provided, align major steps to the methods and weave method names into the step text.",
      "Do not add bracketed method tags in titles.",
      "If method names are not in <language>, translate them into <language> step titles and include the original in parentheses."
    ],
    "kg_guidance": [
      "If kg_fact_cards is provided, use it to ground claims and outline steps.",
      "Prefer higher-confidence and more recent evidence when shaping the outline.",
      "Prefer evidence-backed relationships over unsupported assertions.",
      "Do not copy evidence snippets verbatim; paraphrase.",
      "When you use kg_fact_cards, add a short citation in-line like 'Sources: title1; title2'.",
      "Do not fabricate sources that are not in kg_fact_cards or documents.",
      "Use trend stats and contradictions in kg_fact_cards to highlight agreements, shifts, and disagreements.",
      "Dates in kg_fact_cards come from document added_at timestamps (ingestion time), not publication dates."
    ]
  },
  "kg_fact_cards": ["<fact card>", "..."]
}
```

In [4]:
if RUN_PREFIX:
    attempts = sorted(ARTIFACTS_DIR.glob(RUN_PREFIX + "_outline_attempt*.json"))
else:
    attempts = []
_print_outline_group(attempts, "All Outline Attempts")

=== All Outline Attempts ===
=== 2026_02_25_14_11_09_outline_run_16_outline_attempt1.json ===
File: artifacts/2026_02_25_14_11_09_outline_run_16_outline_attempt1.json
1. 文献综述: 确立AI智能体与工作流自动化基础定义与核心概念
- 检索并综合官方技术报告,学术论文和行业分析,明确AI智能体(AI Agents)的定义,演进阶段及关键特性。这包括理解自主性,反应性,主动性和社会能力等核心属性,为后续研究奠定基础。
- 厘清AI智能体驱动的工作流自动化与传统自动化(如RPA)之间的根本区别。重点关注智能体在复杂决策,动态适应和多步骤任务协调方面的优势,以识别其独特的价值主张。
- 收集并比较不同来源对“AI智能体驱动工作流技术”的关键概念性描述和分类。整合“AI Agents: what they are and how they will completely transform the...”和“Agent Based AI Systems : What is it and How it works?”等文档中的见解,形成统一的理解框架。
2. Systems Engineering: 分析多智能体系统(MAS)的架构与协作机制
- 深入研究多智能体系统(MAS)作为AI智能体工作流核心的架构类型,例如中心化,分布式或混合模型。探究LLM-based多智能体系统如何促进更复杂的智能体间互动与协作,这是新兴的研究领域(Sources: Multi-agent system - Wikipedia; From Solo to Symphony: A Practical Guide to AI Agent Architectures)。
- 解析MAS中智能体之间的通信协议与协调策略,包括A2A Protocol在编排AI智能体中的作用(Source: From Solo to Symphony: A Practical Guide to AI Agent Architectures; Distributed Artificial Intelligence)。分析通信效率,数据传输协议及信息共享机制对系

## Outline Expansions

Input is the best available outline that is too short. Expansion is triggered when the outline word count falls below the minimum length; the model is asked to add detail without removing content, using a length hint derived from the previous outline. Expansion runs after attempts fail length validation and before revision is enforced.

System prompt used by the LLM:
```text
You expand research plan outlines to meet strict minimum length. Return JSON only.
```

User prompt template sent to the LLM:
```json
{
  "previous_outline": ["<outline step>", "..."],
  "instructions": {
    "output_json_schema": {"outline": ["string"]},
    "requirements": [
      "Length must be at least 1000 words in the output language.",
      "Keep 8-12 major steps.",
      "Each major step must include 3-5 substeps.",
      "Each substep should be 2-3 sentences.",
      "Do not remove content; only expand with additional detail and substeps."
    ],
    "language_guidance": [
      "Write natural-language content in <language>.",
      "Keep paper titles, dataset names, benchmarks, model names, APIs, and acronyms in English."
    ],
    "structure_guidance": ["<optional guidance>"]
  },
  "length_hint": "<length hint>",
  "language_hint": "<optional language hint>"
}
```

In [5]:
if RUN_PREFIX:
    expands = sorted(ARTIFACTS_DIR.glob(RUN_PREFIX + "_outline_expand*.json"))
else:
    expands = []
_print_outline_group(expands, "All Outline Expansions")

=== All Outline Expansions ===
=== 2026_02_25_14_11_09_outline_run_16_outline_expand4.json ===
File: artifacts/2026_02_25_14_11_09_outline_run_16_outline_expand4.json
1. Systems Engineering: 界定AI智能体及其工作流自动化核心概念
2. - 进行文献回顾,明确AI智能体(AI Agents)的定义,核心特征及其在工作流自动化中的作用。重点关注其自主性,目标导向行为和环境感知能力,以建立研究的基础理解。这包括深入探讨智能体如何通过感知环境,进行决策并执行动作来达成特定目标,尤其是在动态和复杂的自动化场景中。
3. - 系统分析工作流自动化(Workflow Automation)的概念演变及其与传统自动化的区别。识别AI智能体如何增强现有工作流自动化系统,并确定其主要应用场景。具体将研究智能体如何引入更高级别的适应性和智能,使其能够处理传统自动化系统难以应对的非结构化任务和异常情况。
4. - 深入文献回顾多智能体系统(Multi-Agent Systems, MAS)的基本原理和组成要素,包括代理,环境,交互和组织。理解MAS作为分布式人工智能(DAI)核心研究领域的重要性。本部分将探讨不同类型的MAS架构,以及它们在实现复杂任务协调和解决分布式问题方面的优势。
5. - 区分不同类型的AI智能体,例如反应式智能体,深思式智能体和混合式智能体。系统分析它们在驱动不同复杂度工作流时的适用性及其对工作流设计的影响。研究将侧重于每种智能体类型如何根据其内部结构和推理能力,有效地执行简单到复杂的任务,并探讨其在实际应用中的权衡。
6. Systems Engineering: 识别与分析AI智能体驱动工作流的典型架构与框架
7. - 进行系统分析,识别并分类当前AI智能体驱动工作流中常用的架构模式。重点关注单智能体与多智能体架构的差异及其各自的优缺点。研究将深入探讨这些架构如何影响系统的可扩展性,鲁棒性和整体性能,以及它们在不同应用场景下的最佳实践。
8. - 文献回顾并考察领先的AI智能体框架,例如LangChain,AutoGen,CrewAI等,评估其在构建可扩展工作流方面的能力。

## Outline Revisions

Input is the last outline that still violates length or language constraints. Revision is triggered after attempts and expansions fail to meet the strict word-count target; the model is instructed to rewrite while preserving structure, using a length hint based on the previous outline. Revision occurs near the end of the outline pipeline.

System prompt used by the LLM:
```text
You revise research plan outlines to meet strict length constraints. Return JSON only.
```

User prompt template sent to the LLM:
```json
{
  "previous_outline": ["<outline step>", "..."],
  "instructions": {
    "output_json_schema": {"outline": ["string"]},
    "requirements": [
      "Length must be 1000-2000 words in the output language.",
      "Keep 8-12 major steps.",
      "Each major step must include 3-5 substeps.",
      "Each substep should be 2-3 sentences.",
      "Preserve the original topic coverage and structure; rewrite to fit length."
    ],
    "language_guidance": [
      "Write natural-language content in <language>.",
      "Keep paper titles, dataset names, benchmarks, model names, APIs, and acronyms in English."
    ],
    "structure_guidance": ["<optional guidance>"]
  },
  "length_hint": "<length hint>",
  "language_hint": "<optional language hint>"
}
```

In [6]:
if RUN_PREFIX:
    revisions = sorted(ARTIFACTS_DIR.glob(RUN_PREFIX + "_outline_revision*.json"))
else:
    revisions = []
_print_outline_group(revisions, "All Outline Revisions")

=== All Outline Revisions ===
=== 2026_02_25_14_11_09_outline_run_16_outline_revision6.json ===
File: artifacts/2026_02_25_14_11_09_outline_run_16_outline_revision6.json
1. Systems Engineering: 界定AI智能体及其工作流自动化核心概念
2. 本研究将进行全面的文献回顾,明确AI智能体(AI Agents)的精确定义,核心特征及其在工作流自动化中的关键作用。我们将深入探讨智能体感知环境,自主决策和执行动作的基础机制,尤其关注其在动态复杂自动化场景中的适应能力。
3. 系统分析工作流自动化(Workflow Automation)的概念演变及其与传统自动化的区别。识别AI智能体如何通过引入智能和适应性,增强现有工作流系统,从而处理传统自动化难以应对的非结构化任务和异常情况。
4. 深入回顾多智能体系统(Multi-Agent Systems, MAS)的基本原理和组成要素,包括代理,环境,交互和组织。理解MAS作为分布式人工智能(DAI)核心研究领域的重要性,强调其在解决复杂分布式问题上的独特优势。
5. 区分反应式,深思式和混合式AI智能体,并系统分析它们在驱动不同复杂程度工作流时的适用性。研究将侧重于每种智能体如何根据其内部结构和推理能力执行任务,并权衡实际应用中性能,资源消耗和响应速度。
6. Systems Engineering: 识别与分析AI智能体驱动工作流的典型架构与框架
7. 开展系统分析,识别并分类AI智能体驱动工作流中常用的架构模式。重点关注单智能体与多智能体架构的差异及其优缺点,尤其在处理并发任务和分布式决策方面的表现。
8. 文献回顾并考察领先的AI智能体框架,例如LangChain,AutoGen和CrewAI,评估其构建可扩展工作流的能力。分析这些框架如何支持任务分解,工具使用,记忆和状态管理,这些功能对复杂自动化至关重要。
9. 深入探索基于大型语言模型(LLM)的多智能体系统架构,理解LLM如何作为智能体的核心决策引擎。系统分析LLM在实现智能体间复杂交互和协调中的作用,包括其在语义理解和生成复杂指令方面的能力。
10. 系统分析智能体架构中的关键

## Plan Draft & Refinement Artifacts

These artifacts capture the parsed plan fields after the draft and refinement steps in each round.
They are saved as JSON in `artifacts/` with filenames like `*_plan_run_*_plan_draft_round*.json` and `*_plan_run_*_plan_refine_round*.json`.

System prompt used by the LLM:
```text
You are a research planning agent. Produce a plan that is concise, actionable, and focused on gaps. All natural-language content MUST be in the requested output language.
Return ONE valid JSON object only. No markdown, no commentary, no extra text.

Requirements:
- Must include keys: scope, key_questions, keywords, gaps, notes, readiness
- scope, key_questions, keywords MUST be non-empty arrays (>= 3 items each)
- keywords must include core terms from the topic and interests; include extracted interests if provided
- methods are analysis approaches; use them to frame the plan, not as the topic itself
- gaps and notes may be empty arrays
- retrieval_queries is optional; if provided, it must be a short list of search phrases
- readiness must be one of: "draft", "refined", "ready"
- Output must be strict JSON (double quotes, no trailing commas)
- All natural-language content must be in the requested output language.
```


In [ ]:
plan_prefix = _pick_plan_prefix(RUN_PREFIX)
print(f"PLAN_PREFIX: {plan_prefix}")

drafts = []
refines = []
if plan_prefix:
    drafts = sorted(ARTIFACTS_DIR.glob(plan_prefix + "_plan_draft_round*.json"))
    refines = sorted(ARTIFACTS_DIR.glob(plan_prefix + "_plan_refine_round*.json"))
else:
    drafts = sorted(ARTIFACTS_DIR.glob("*_plan_draft_round*.json"), key=lambda p: p.stat().st_mtime)
    refines = sorted(ARTIFACTS_DIR.glob("*_plan_refine_round*.json"), key=lambda p: p.stat().st_mtime)
    if AUTO_PICK_LATEST:
        drafts = drafts[-1:]
        refines = refines[-1:]

_print_plan_group(drafts, "Plan Draft Artifacts")
_print_plan_group(refines, "Plan Refinement Artifacts")


## Plan Artifact

Input is the finalized plan fields and the selected outline. The plan is triggered at the end of the research run and rendered into markdown by `render_plan_md()` after the LLM produces plan fields. Plan generation happens before outline building and refinement rounds if needed.

System prompt used by the LLM:
```text
You are a research planning agent. Produce a plan that is concise, actionable, and focused on gaps. All natural-language content MUST be in the requested output language.
Return ONE valid JSON object only. No markdown, no commentary, no extra text.

Requirements:
- Must include keys: scope, key_questions, keywords, gaps, notes, readiness
- scope, key_questions, keywords MUST be non-empty arrays (>= 3 items each)
- keywords must include core terms from the topic and interests; include extracted interests if provided
- methods are analysis approaches; use them to frame the plan, not as the topic itself
- gaps and notes may be empty arrays
- retrieval_queries is optional; if provided, it must be a short list of search phrases
- readiness must be one of: "draft", "refined", "ready"
- Output must be strict JSON (double quotes, no trailing commas)
- All natural-language content MUST be in the requested output language, except retrieval_queries must be in English.
- Keep JSON keys in English.
- Keep paper titles, dataset names, benchmarks, model names, APIs, and acronyms in English.
- Use ASCII punctuation for JSON (":" and ","), not full-width punctuation.
- When evidence includes confidence or recency, prefer higher-confidence and more recent evidence.
All natural-language content MUST be in the requested output language.

Example (Chinese):
{"scope":["..."],"key_questions":["..."],"keywords":["..."],"gaps":["..."],"notes":["..."],"readiness":"draft","retrieval_queries":["..."]}

Example (English):
{"scope":["..."],"key_questions":["..."],"keywords":["..."],"gaps":["..."],"notes":["..."],"readiness":"draft","retrieval_queries":["..."]}
```

User prompt template sent to the LLM:
```json
{
  "topic": "<topic>",
  "interests": ["<interest>", "..."],
  "methods": ["<method>", "..."],
  "documents": [
    {"title": "<title>", "source_type": "<type>", "source": "<source>", "doc_id": "<id>"}
  ],
  "output_language": "<language>",
  "instructions": {
    "output_json_schema": {
      "scope": ["string"],
      "key_questions": ["string"],
      "keywords": ["string"],
      "gaps": ["string"],
      "notes": ["string"],
      "retrieval_queries": ["string"],
      "readiness": "draft | refined | ready"
    },
    "language_policy": [
      "All list items must be in <language>.",
      "retrieval_queries must be in English.",
      "Keep paper titles, dataset names, benchmarks, model names, APIs, and acronyms in English."
    ],
    "keyword_guidance": [
      "Include key terms from the topic.",
      "Include salient terms from manually added interests.",
      "If extracted_interests are provided, incorporate them.",
      "Do not replace the topic with method names."
    ],
    "retrieval_query_guidance": [
      "If provided, use 2-6 word search phrases.",
      "Avoid negations like 'no' or 'lack of'.",
      "Focus on entities, relationships, and concrete nouns.",
      "Keep to 2-8 total queries.",
      "Queries must be in English."
    ],
    "method_guidance": [
      "Methods are analysis approaches (lenses), not standalone topics.",
      "Apply methods to the topic and evidence."
    ],
    "kg_guidance": [
      "If kg_fact_cards is provided, use it to ground claims and plan steps.",
      "Prefer higher-confidence and more recent evidence when shaping the plan.",
      "Do not copy evidence snippets verbatim; paraphrase.",
      "Do not fabricate sources that are not in kg_fact_cards or documents."
    ]
  },
  "extracted_interests": ["<extracted>", "..."],
  "kg_fact_cards": ["<fact card>", "..."],
  "skill_guidance": ["<skill guidance>"]
}
```

In [8]:
plan = None
plan_prefix = None
if RUN_TIMESTAMP:
    plan_prefix = RUN_TIMESTAMP[:16]  # YYYY_MM_DD_HH_MM
elif RUN_PREFIX:
    match = re.match(r"^(\d{4}_\d{2}_\d{2}_\d{2}_\d{2}_\d{2})", RUN_PREFIX)
    if match:
        plan_prefix = match.group(1)[:16]

if plan_prefix:
    candidates = list(ARTIFACTS_DIR.glob(plan_prefix + "_plan.md"))
    if candidates:
        plan = max(candidates, key=lambda p: p.stat().st_mtime)
if not plan:
    candidates = list(ARTIFACTS_DIR.glob("*_plan.md"))
    if candidates:
        plan = max(candidates, key=lambda p: p.stat().st_mtime)

_print_markdown(plan, "Plan Artifact")


=== Plan Artifact ===
File: artifacts/2026_02_25_14_19_plan.md
# Research Plan - AI agent-driven workflow technologies

- Created at: 2026-02-25T19:10:52.941341+00:00
- Round: 2
- Readiness: refined

## Scope
- 研究AI智能体驱动工作流技术的核心概念,定义和演进,特别关注基于大型语言模型(LLM)的多智能体系统(MAS)。
- 分析基于LLM的多智能体系统在复杂工作流自动化中的主要架构,设计模式,编排与通信机制(如A2A协议),以及它们如何提升协作智能和实现自组织能力。
- 探讨AI智能体驱动工作流在不同行业(例如金融,医疗,在线交易)应用中的潜力,当前趋势及未来发展方向。
- 评估AI智能体驱动工作流,尤其是LLM驱动的多智能体系统在实际部署中面临的技术,安全(包括MAS中的风险升级和防火墙架构)与伦理挑战。

## Key Questions
- 当前AI智能体驱动工作流技术,特别是基于LLM的多智能体系统,有哪些主要发展趋势,新兴架构以及它们如何促进更复杂的智能体交互？
- 基于LLM的多智能体系统如何在复杂工作流自动化和高级协作智能方面运作？其关键的编排,通信(如A2A协议)和自组织机制是什么？
- 部署AI智能体驱动工作流(特别是MAS)面临哪些关键技术,安全(如多智能体系统中的安全风险升级)和可信赖性挑战,以及如何通过系统工程方法和拟议的防火墙架构来缓解这些挑战？

## Keywords
- AI智能体, 工作流自动化, 多智能体系统 (MAS), 智能体架构, 协作智能, 分布式人工智能 (DAI), LLM-based多智能体系统, 工作流编排, 安全性, 部署趋势, cmbagent, A2A协议, 自组织

## Source Types in Selected Docs
- web

## Source Types in Library
- file: 10
- web: 29

## Gaps and Retrieval Needs
- 缺乏关于AI智能体驱动工作流在特定行业(如金融,医疗)中实际效益和投资回报率(ROI)的深入案例研